# PII Detection using Regex
## Testing on context.tsv with validation against val.csv

This notebook uses regex patterns to detect Personally Identifiable Information (PII) in Arabic text.

In [1]:
import pandas as pd
import re
from collections import defaultdict

# Load the data
context_df = pd.read_csv('res/data/context.tsv', sep='\t')
val_df = pd.read_csv('res/data/val.csv')

print(f"Context rows: {len(context_df)}")
print(f"Validation rows: {len(val_df)}")
print(f"\nColumns in val.csv:")
print(val_df.columns.tolist())

Context rows: 110
Validation rows: 110

Columns in val.csv:
['Credit Card Nos', 'Email Addresses', 'IP Addresses', 'IBAN or Swift Codes', 'Passport Numbers', 'MAC Addresses', 'Phone Number', 'National ID Number', 'FirstName', 'FatherName', 'GrandFatherName', 'FamilyName', 'Street Name', 'Username', 'Postal Code', 'License Plate', 'Medical History', 'Biometric Data', 'Job Title', 'date_of_birth', 'اسم الشارع', 'التاريخ الطبي', 'البيانات الحيوية', 'المسمى الوظيفي']


## Define Regex Patterns for PII Detection

In [2]:
# Define regex patterns for each PII type
patterns = {
    'Credit Card Nos': r'\b\d{4}[\s-]\d{4}[\s-]\d{4}[\s-]\d{4}\b',
    'Email Addresses': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
    'IP Addresses': r'\b(?:\d{1,3}\.){3}\d{1,3}\b',
    'IBAN or Swift Codes': r'\b[A-Z]{2}\d{2}[A-Z0-9]{4,30}\b',
    'Passport Numbers': r'\b[A-Z]?\d{6,9}\b',
    'MAC Addresses': r'\b(?:[0-9A-Fa-f]{2}:){5}[0-9A-Fa-f]{2}\b',
    'Phone Number': r'\b(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{3,4}\b',
    'National ID Number': r'\b\d{7,10}\b',
    'Username': r'\b[A-Za-z0-9_]{3,20}\b',
    'Postal Code': r'\b\d{5}\b',
    'License Plate': r'\b[A-Z]{3}\s?\d{4}\b',
    'date_of_birth': r'\b\d{4}[/\-]\d{2}[/\-]\d{2}\b',
}

# Compile patterns
compiled_patterns = {name: re.compile(pattern) for name, pattern in patterns.items()}

print("Regex patterns defined for:", list(patterns.keys()))

Regex patterns defined for: ['Credit Card Nos', 'Email Addresses', 'IP Addresses', 'IBAN or Swift Codes', 'Passport Numbers', 'MAC Addresses', 'Phone Number', 'National ID Number', 'Username', 'Postal Code', 'License Plate', 'date_of_birth']


In [3]:
# Arabic names patterns (FirstName, FatherName, GrandFatherName, FamilyName)
# These are detected from context text in square brackets

def extract_arabic_names(text):
    """Extract Arabic names from square brackets"""
    # Find all content in square brackets
    brackets = re.findall(r'\[(.*?)\]', text)
    return brackets

def detect_pii_regex(text):
    """Detect all PII types using regex"""
    results = defaultdict(list)
    
    # Detect Credit Card Numbers
    results['Credit Card Nos'] = compiled_patterns['Credit Card Nos'].findall(text)
    
    # Detect Email Addresses
    results['Email Addresses'] = compiled_patterns['Email Addresses'].findall(text)
    
    # Detect IP Addresses
    ip_pattern = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')
    results['IP Addresses'] = ip_pattern.findall(text)
    
    # Detect IBAN/Swift Codes
    iban_pattern = re.compile(r'\b[A-Z]{2}\d{2}[A-Z0-9]{4,30}\b')
    results['IBAN or Swift Codes'] = iban_pattern.findall(text)
    
    # Detect Passport Numbers (single letter followed by 6-9 digits)
    passport_pattern = re.compile(r'\b[A-Z]\d{6,9}\b')
    results['Passport Numbers'] = passport_pattern.findall(text)
    
    # Detect MAC Addresses
    results['MAC Addresses'] = compiled_patterns['MAC Addresses'].findall(text)
    
    # Detect Phone Numbers
    phone_pattern = re.compile(r'\+\d{1,3}[\s.-]?\d{2,4}[\s.-]?\d{3,4}[\s.-]?\d{3,4}')
    results['Phone Number'] = phone_pattern.findall(text)
    
    # Detect National ID (7-10 digit numbers that aren't credit cards or phone numbers)
    national_id_pattern = re.compile(r'\b\d{7,10}\b')
    # Filter out other patterns that might look like national IDs
    all_nums = national_id_pattern.findall(text)
    filtered_nids = []
    for num in all_nums:
        # Skip if it looks like part of a credit card or phone
        if len(num) >= 7 and len(num) <= 10:
            filtered_nids.append(num)
    results['National ID Number'] = filtered_nids
    
    # Extract content in square brackets for names and other fields
    brackets_content = extract_arabic_names(text)
    
    # Simple names are Arabic words (not containing Latin chars or numbers)
    arabic_name_pattern = re.compile(r'^[\u0600-\u06FF]+$')
    
    # Count occurrences of each PII type
    return results, brackets_content

# Test on first row
test_text = context_df.iloc[0]['text']
results, brackets = detect_pii_regex(test_text)
print("Sample detection results:")
for key, values in results.items():
    if values:
        print(f"  {key}: {values[:3]}...")

Sample detection results:
  Credit Card Nos: ['6011-0000-5555-4444']...
  Email Addresses: ['chris.evans@marvel.com']...
  IP Addresses: ['192.168.15.15']...
  Passport Numbers: ['E1234567']...
  MAC Addresses: ['BB:CC:DD:EE:FF:00']...
  National ID Number: ['4343434343']...


In [4]:
# Map columns from val.csv to regex-detectable categories
regex_columns = [
    'Credit Card Nos',
    'Email Addresses',
    'IP Addresses',
    'IBAN or Swift Codes',
    'Passport Numbers',
    'MAC Addresses',
    'Phone Number',
    'National ID Number',
    'FirstName',
    'FatherName',
    'GrandFatherName',
    'FamilyName',
    'Street Name',
    'Username',
    'Postal Code',
    'License Plate',
    'Medical History',
    'Biometric Data',
    'Job Title',
    'date_of_birth',
]

def has_pii_regex(text, pii_type):
    """Check if text has a specific PII type using regex"""
    if pii_type == 'Credit Card Nos':
        return bool(compiled_patterns['Credit Card Nos'].search(text))
    elif pii_type == 'Email Addresses':
        return bool(compiled_patterns['Email Addresses'].search(text))
    elif pii_type == 'IP Addresses':
        return bool(re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b').search(text))
    elif pii_type == 'IBAN or Swift Codes':
        return bool(re.compile(r'\b[A-Z]{2}\d{2}[A-Z0-9]{4,30}\b').search(text))
    elif pii_type == 'Passport Numbers':
        return bool(re.compile(r'\b[A-Z]\d{6,9}\b').search(text))
    elif pii_type == 'MAC Addresses':
        return bool(compiled_patterns['MAC Addresses'].search(text))
    elif pii_type == 'Phone Number':
        return bool(re.compile(r'\+\d{1,3}[\s.-]?\d{2,4}[\s.-]?\d{3,4}[\s.-]?\d{3,4}').search(text))
    elif pii_type == 'National ID Number':
        # Look for 7-10 digit numbers
        return bool(re.compile(r'\b\d{7,10}\b').search(text))
    elif pii_type == 'FirstName':
        # Arabic names in brackets (first name position)
        brackets = re.findall(r'\[(.*?)\]', text)
        # First, Father, GrandFather, Family names appear in sequence
        return len(brackets) >= 1
    elif pii_type == 'FatherName':
        brackets = re.findall(r'\[(.*?)\]', text)
        return len(brackets) >= 2
    elif pii_type == 'GrandFatherName':
        brackets = re.findall(r'\[(.*?)\]', text)
        return len(brackets) >= 3
    elif pii_type == 'FamilyName':
        brackets = re.findall(r'\[(.*?)\]', text)
        return len(brackets) >= 4
    elif pii_type == 'Street Name':
        # Streets contain 'Rd' or 'St' or Arabic road keywords
        return bool(re.search(r'\b(Rd|St|Street|Road)\b', text, re.IGNORECASE) or 
                    bool(re.search(r'طريق', text)))
    elif pii_type == 'Username':
        # usernames like 'saad_v', 'khalid_f'
        return bool(re.compile(r'\b[A-Za-z0-9_]{3,15}\b').search(text))
    elif pii_type == 'Postal Code':
        return bool(re.compile(r'\b\d{5}\b').search(text))
    elif pii_type == 'License Plate':
        return bool(re.compile(r'\b[A-Z]{3}\s?\d{4}\b').search(text))
    elif pii_type == 'Medical History':
        return bool(re.search(r'سجل طبي|صحة|تاريخ مرضي|حالة صحية| Allergies|health', text, re.IGNORECASE))
    elif pii_type == 'Biometric Data':
        return bool(re.search(r'بصمة|بصمة الإصبع|Fingerprint|Biometric|Face ID|Iris|Touch ID', text, re.IGNORECASE))
    elif pii_type == 'Job Title':
        return bool(re.search(r'مهندس|محاسب|طبيب|مدير|موظف|كاتب|مهندس معماري|Biologist|Engineer|Manager|Accountant', text, re.IGNORECASE))
    elif pii_type == 'date_of_birth':
        return bool(re.compile(r'\b\d{4}[/\-]\d{2}[/\-]\d{2}\b').search(text))
    return False

print("PII detection functions defined successfully")

PII detection functions defined successfully


In [5]:
# Process all rows and compare regex detection with validation labels
correct = defaultdict(int)
total = defaultdict(int)
false_positives = defaultdict(int)
false_negatives = defaultdict(int)

for idx in range(len(context_df)):
    text = context_df.iloc[idx]['text']
    
    for col in regex_columns:
        # Get ground truth from val.csv (row idx starts at 2 because row 1 is header, row 2 is first data)
        actual = val_df.iloc[idx][col] == 1
        predicted = has_pii_regex(text, col)
        
        total[col] += 1
        if actual == predicted:
            correct[col] += 1
        else:
            if predicted and not actual:
                false_positives[col] += 1
            elif actual and not predicted:
                false_negatives[col] += 1

print("Accuracy per PII type:")
print("-" * 60)
for col in regex_columns:
    accuracy = correct[col] / total[col] * 100 if total[col] > 0 else 0
    print(f"{col:25s}: {accuracy:6.2f}% ({correct[col]}/{total[col]})")
    print(f"    False Positives: {false_positives[col]}, False Negatives: {false_negatives[col]}")

Accuracy per PII type:
------------------------------------------------------------
Credit Card Nos          :  89.09% (98/110)
    False Positives: 1, False Negatives: 11
Email Addresses          : 100.00% (110/110)
    False Positives: 0, False Negatives: 0
IP Addresses             : 100.00% (110/110)
    False Positives: 0, False Negatives: 0
IBAN or Swift Codes      :  48.18% (53/110)
    False Positives: 0, False Negatives: 57
Passport Numbers         : 100.00% (110/110)
    False Positives: 0, False Negatives: 0
MAC Addresses            : 100.00% (110/110)
    False Positives: 0, False Negatives: 0
Phone Number             :  70.00% (77/110)
    False Positives: 0, False Negatives: 33
National ID Number       :  97.27% (107/110)
    False Positives: 3, False Negatives: 0
FirstName                :  58.18% (64/110)
    False Positives: 46, False Negatives: 0
FatherName               :  61.82% (68/110)
    False Positives: 41, False Negatives: 1
GrandFatherName          :  68.18% (

In [6]:
# Overall statistics
total_predictions = sum(total.values())
total_correct = sum(correct.values())
overall_accuracy = total_correct / total_predictions * 100 if total_predictions > 0 else 0

print(f"\nOverall Accuracy: {overall_accuracy:.2f}%")
print(f"Total predictions: {total_predictions}")
print(f"Correct predictions: {total_correct}")


Overall Accuracy: 81.68%
Total predictions: 2200
Correct predictions: 1797


In [7]:
# Show example matches for each PII type
print("\n" + "="*80)
print("Example PII matches from context.tsv")
print("="*80)

for pii_type in regex_columns:
    print(f"\n{pii_type}:")
    found = False
    for idx in range(min(50, len(context_df))):
        text = context_df.iloc[idx]['text']
        if has_pii_regex(text, pii_type):
            matches = []
            if pii_type == 'Credit Card Nos':
                matches = compiled_patterns['Credit Card Nos'].findall(text)
            elif pii_type == 'Email Addresses':
                matches = compiled_patterns['Email Addresses'].findall(text)
            elif pii_type == 'IP Addresses':
                matches = re.findall(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', text)
            elif pii_type == 'IBAN or Swift Codes':
                matches = re.findall(r'\b[A-Z]{2}\d{2}[A-Z0-9]{4,30}\b', text)
            elif pii_type == 'MAC Addresses':
                matches = compiled_patterns['MAC Addresses'].findall(text)
            elif pii_type == 'Phone Number':
                matches = re.findall(r'\+\d{1,3}[\s.-]?\d{2,4}[\s.-]?\d{3,4}[\s.-]?\d{3,4}', text)
            elif pii_type == 'date_of_birth':
                matches = re.findall(r'\d{4}[/\-]\d{2}[/\-]\d{2}', text)
            elif pii_type == 'National ID Number':
                matches = re.findall(r'\b\d{7,10}\b', text)
            elif pii_type in ['FirstName', 'FatherName', 'GrandFatherName', 'FamilyName']:
                brackets = re.findall(r'\[(.*?)\]', text)
                matches = brackets[:4] if brackets else []
            elif pii_type == 'License Plate':
                matches = re.findall(r'\b[A-Z]{3}\s?\d{4}\b', text)
            elif pii_type == 'Street Name':
                matches = re.findall(r'[A-Za-z]+(?:\s+(?:Rd|St|Street|Road))', text)[:3]
            elif pii_type == 'Medical History':
                matches = re.findall(r'[\u0600-\u06FF]+(?:\s+سجل|\s+صحة|\s+تاريخ|\s+حالة)', text)[:3]
            elif pii_type == 'Biometric Data':
                matches = re.findall(r'(?:بصمة|Fingerprint|Biometric|Face ID|Touch ID)', text)[:3]
            elif pii_type == 'Job Title':
                matches = re.findall(r'(?:مهندس|محاسب|طبيب|مدير|Biologist|Engineer|Manager)', text)[:3]
            
            if matches:
                print(f"  Row {idx+2}: {matches}")
                found = True
                break
    if not found:
        print("  (No examples found)")


Example PII matches from context.tsv

Credit Card Nos:
  Row 2: ['6011-0000-5555-4444']

Email Addresses:
  Row 2: ['chris.evans@marvel.com']

IP Addresses:
  Row 2: ['192.168.15.15']

IBAN or Swift Codes:
  (No examples found)

Passport Numbers:
  (No examples found)

MAC Addresses:
  Row 2: ['BB:CC:DD:EE:FF:00']

Phone Number:
  Row 5: ['+1 345-555-0101']

National ID Number:
  Row 2: ['4343434343']

FirstName:
  Row 2: ['6011-0000-5555-4444', 'chris.evans@marvel.com', '192.168.15.15', 'US00 JPM 1234 5678 90']

FatherName:
  Row 2: ['6011-0000-5555-4444', 'chris.evans@marvel.com', '192.168.15.15', 'US00 JPM 1234 5678 90']

GrandFatherName:
  Row 2: ['6011-0000-5555-4444', 'chris.evans@marvel.com', '192.168.15.15', 'US00 JPM 1234 5678 90']

FamilyName:
  Row 2: ['6011-0000-5555-4444', 'chris.evans@marvel.com', '192.168.15.15', 'US00 JPM 1234 5678 90']

Street Name:
  Row 3: ['Thumamah Rd']

Username:
  (No examples found)

Postal Code:
  (No examples found)

License Plate:
  Row 2: [

In [8]:
# Create a detailed confusion matrix for each PII category
import pandas as pd

confusion_data = []
for col in regex_columns:
    confusion_data.append({
        'PII Type': col,
        'True Positives': correct[col],
        'False Positives': false_positives[col],
        'False Negatives': false_negatives[col],
        'Total': total[col],
        'Accuracy': f"{correct[col] / total[col] * 100:.1f}%" if total[col] > 0 else 'N/A'
    })

confusion_df = pd.DataFrame(confusion_data)
print(confusion_df.to_string(index=False))

           PII Type  True Positives  False Positives  False Negatives  Total Accuracy
    Credit Card Nos              98                1               11    110    89.1%
    Email Addresses             110                0                0    110   100.0%
       IP Addresses             110                0                0    110   100.0%
IBAN or Swift Codes              53                0               57    110    48.2%
   Passport Numbers             110                0                0    110   100.0%
      MAC Addresses             110                0                0    110   100.0%
       Phone Number              77                0               33    110    70.0%
 National ID Number             107                3                0    110    97.3%
          FirstName              64               46                0    110    58.2%
         FatherName              68               41                1    110    61.8%
    GrandFatherName              75               35  

In [9]:
# Precision and Recall per category
print("\n" + "="*80)
print("Precision and Recall Analysis")
print("="*80)

for col in regex_columns:
    tp = correct[col]
    fp = false_positives[col]
    fn = false_negatives[col]
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\n{col}:")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")


Precision and Recall Analysis

Credit Card Nos:
  Precision: 0.9899
  Recall:    0.8991
  F1 Score:  0.9423

Email Addresses:
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000

IP Addresses:
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000

IBAN or Swift Codes:
  Precision: 1.0000
  Recall:    0.4818
  F1 Score:  0.6503

Passport Numbers:
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000

MAC Addresses:
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000

Phone Number:
  Precision: 1.0000
  Recall:    0.7000
  F1 Score:  0.8235

National ID Number:
  Precision: 0.9727
  Recall:    1.0000
  F1 Score:  0.9862

FirstName:
  Precision: 0.5818
  Recall:    1.0000
  F1 Score:  0.7356

FatherName:
  Precision: 0.6239
  Recall:    0.9855
  F1 Score:  0.7640

GrandFatherName:
  Precision: 0.6818
  Recall:    1.0000
  F1 Score:  0.8108

FamilyName:
  Precision: 0.7870
  Recall:    0.9770
  F1 Score:  0.8718

Street Name:
  Precision: 0.8818
  Recall:    1.